In [11]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sys
from PIL.ImageColor import colormap
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from scipy.integrate import solve_ivp
from scipy.optimize import curve_fit
from pathlib import Path

project_root = next((p for p in [Path.cwd(), *Path.cwd().parents] if p.name == "discretize"), None)
if project_root is None:
    raise FileNotFoundError("Could not locate project root folder named 'discretize'.")

print("Project root path:")
print(project_root)

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from configuration.settings import *
from model.fuelcell import PEMFC_1D
from model.coefficients import *
from model.states import *
from configuration.sim_initialize import *

prj_root = project_root.as_posix() + "/"
dataset_path = project_root / "data" / "polarization"
info_data = pd.read_excel(dataset_path / "DATAFILES_LIST.xlsx")

Project root path:
c:\Users\ykh_w\Dropbox\PhD folder\Codes\dual-scale model\discretize


In [12]:
i = 0.4e4 # A.m-2
i0 = 1.2e-1 # A.m-2

##### Source of the heat
###### CL: entropy change + activation + resistive + phase change

In [13]:
# Entropy change = i * delta_S * T / nF 
# For HOR, delta_S = 0.104 # J/(mol*K)
# For OOR, delta_S = -163.3 #  J/(mol*K)
S_H_OOR = i * (-163.3) * 333.15 / (2 * 96485) # W.m-2
S_H_HOR = i * (-0.104) * 333.15 / (2 * 96485)     # W.m-2

In [14]:
# Activation source = i * eta
S_act = i * 0.3

In [15]:
# resistance heat source = i^2 * R
S_re = i**2 * 4e-6

In [16]:
# Phase change heat source 
S_phase = gamma_sorp(C_v=5, s=0.05, lambdaa=10, T=333.15, Hcl=1e-6, Kshape=1) *42e3

In [21]:
gamma_sorp(C_v=5, s=0.05, lambdaa=10, T=333.15, Hcl=1e-6, Kshape=1) 

np.float64(13.903533625757413)

In [20]:
0.4e4 / (2*96485)

0.020728610664870188

In [17]:
print("Entropy change for HOR: ", S_H_HOR, " W.m-2")
print("Entropy change for OOR: ", S_H_OOR, " W.m-2")
print("Activation source: ", S_act, " W.m-2")
print("Resistance heat source: ", S_re, " W.m-2")
print("Phase change heat source: ", S_phase, " W.m-2")

Entropy change for HOR:  -0.7181966108721562  W.m-2
Entropy change for OOR:  -1127.7067938021455  W.m-2
Activation source:  1200.0  W.m-2
Resistance heat source:  64.0  W.m-2
Phase change heat source:  583948.4122818114  W.m-2
